# NSBH KN-positive parameter distribution (wide prior)

Objective:
- Visualize which NSBH parameter regions can produce KN.
- Use a broad prior instead of `injections.dat`.

KN-positive definition (same as `KN_params_nsbh.ipynb`):
- `mej_dyn > 0` and `mej_wind > 0`.

Sync note:
- The formulae here are aligned with
  `gw-kn-multimodal/dataset/O5_sim_nsbh_aug/KN_params_nsbh.ipynb`.


In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.interpolate import interp1d

try:
    from astropy.table import Table
    HAS_ASTROPY = True
except Exception:
    HAS_ASTROPY = False

plt.style.use('default')

# Sampling configuration
RNG_SEED = 42
N_CANDIDATES = 1_000_000
SAMPLING_MODE = 'uniform'   # 'uniform' or 'stratified'

BH_MASS_RANGE = (2.0, 20.0)
NS_MASS_RANGE = (1.0, 2.0)
BH_SPIN_MAX = 0.99
NS_SPIN_MAX = 0.5

# KN-positive thresholds on ejecta components
MEJ_DYN_THRESHOLD = 0.0
MEJ_WIND_THRESHOLD = 0.0

# Files
EOS_FILE = Path('eos.mr')
OUTPUT_H5 = Path('nsbh_training.h5')

rng = np.random.default_rng(RNG_SEED)
np.random.seed(RNG_SEED)

print(f'RNG seed: {RNG_SEED}')
print(f'N candidates: {N_CANDIDATES:,}')
print(f'Sampling mode: {SAMPLING_MODE}')
print(f'mej_dyn threshold: {MEJ_DYN_THRESHOLD}')
print(f'mej_wind threshold: {MEJ_WIND_THRESHOLD}')
print(f'EOS file: {EOS_FILE.resolve()}')


## Physics model (same as `KN_params_nsbh.ipynb`)

- NS compactness is computed from EOS-interpolated NS radius.
- Ejecta model includes three components:
  - total ejecta mass `mej_tot`
  - dynamical ejecta mass `mej_dyn`
  - wind ejecta mass `mej_wind`


In [ ]:
def load_eos_interpolator(eos_file: Path):
    eos = np.loadtxt(eos_file, skiprows=1)
    radius = eos[:, 0]
    mass = eos[:, 1]

    order = np.argsort(mass)
    mass = mass[order]
    radius = radius[order]

    return interp1d(mass, radius, kind='cubic', fill_value='extrapolate')


def sample_uniform(rng: np.random.Generator, low: float, high: float, n: int) -> np.ndarray:
    return rng.uniform(low, high, n)


def sample_stratified_1d(rng: np.random.Generator, low: float, high: float, n: int) -> np.ndarray:
    u = (np.arange(n) + rng.random(n)) / n
    rng.shuffle(u)
    return low + (high - low) * u


def draw_candidates(n_draw: int):
    sample_fn = sample_uniform if SAMPLING_MODE == 'uniform' else sample_stratified_1d

    mass1 = sample_fn(rng, BH_MASS_RANGE[0], BH_MASS_RANGE[1], n_draw)
    mass2 = sample_fn(rng, NS_MASS_RANGE[0], NS_MASS_RANGE[1], n_draw)
    spin1z = sample_fn(rng, -BH_SPIN_MAX, BH_SPIN_MAX, n_draw)
    spin2z = sample_fn(rng, -NS_SPIN_MAX, NS_SPIN_MAX, n_draw)

    # Ensure convention mass1 >= mass2
    swap = mass1 < mass2
    if np.any(swap):
        mass1[swap], mass2[swap] = mass2[swap].copy(), mass1[swap].copy()
        spin1z[swap], spin2z[swap] = spin2z[swap].copy(), spin1z[swap].copy()

    return mass1, mass2, spin1z, spin2z


def compute_compactness(mass: np.ndarray, radius_km: np.ndarray) -> np.ndarray:
    g_const = 6.67430e-11
    c_light = 299792458.0
    solar_mass = 1.98855e30
    return (g_const * mass * solar_mass) / (c_light**2 * radius_km * 1000.0)


def compute_R_ISCO(spinz_bh: np.ndarray) -> np.ndarray:
    z1 = 1 + (1 - spinz_bh**2) ** (1 / 3) * ((1 + spinz_bh) ** (1 / 3) + (1 - spinz_bh) ** (1 / 3))
    z2 = np.sqrt(3 * spinz_bh**2 + z1**2)
    return 3 + z2 - np.sign(spinz_bh) * np.sqrt((3 - z1) * (3 + z1 + 2 * z2))


def total_ejecta_mass(mass_ns: np.ndarray, mass_bh: np.ndarray, c_ns: np.ndarray, spinz_bh: np.ndarray) -> np.ndarray:
    a = 0.406
    b = 0.139
    gamma = 0.255
    delta = 1.761

    q = mass_ns / mass_bh
    eta = (1 + 1 / q) ** (-2) * q ** (-1)

    r_isco = compute_R_ISCO(spinz_bh)
    mass_b_ns = mass_ns * (1 + 0.6 * c_ns / (1 - 0.5 * c_ns))

    term = a * (1 - 2 * c_ns) / eta ** (1 / 3) - b * r_isco * c_ns / eta + gamma
    return mass_b_ns * np.where(term > 0, term, 0) ** delta


def compute_dynamical_ejecta_mass(mass_ns: np.ndarray, mass_bh: np.ndarray, spinz_bh: np.ndarray, c_ns: np.ndarray) -> np.ndarray:
    a1 = 0.007116
    a2 = 0.001436
    a4 = -0.02762
    n1 = 0.8636
    n2 = 1.6840

    r_isco = compute_R_ISCO(spinz_bh)
    q = mass_ns / mass_bh
    mass_b_ns = mass_ns * (1 + 0.6 * c_ns / (1 - 0.5 * c_ns))

    mej_dyn = mass_b_ns * (a1 * q ** (-n1) * (1 - 2 * c_ns) / c_ns - a2 * q ** (-n2) * r_isco + a4)
    return mej_dyn


def compute_wind_ejecta_mass(mej_tot: np.ndarray, mej_dyn: np.ndarray, q: np.ndarray) -> np.ndarray:
    # This follows KN_params_nsbh.ipynb exactly (single random scaling factor).
    mass_disc = np.where(mej_tot - mej_dyn > 0, mej_tot - mej_dyn, 0)
    xi = 0.18 + 0.11 / (1 + np.e ** (1.5 * (1 / q - 3)))
    mej_th = xi * mass_disc
    mej_mag = np.random.uniform(0.1, 1.0) * mej_th
    return mej_th + mej_mag


## Draw broad-prior candidates and classify KN-positive events

Classification rule:
- `kn_positive = (mej_dyn > MEJ_DYN_THRESHOLD) & (mej_wind > MEJ_WIND_THRESHOLD)`.


In [ ]:
radius_interp = load_eos_interpolator(EOS_FILE)

mass1, mass2, spin1z, spin2z = draw_candidates(N_CANDIDATES)
radius2 = radius_interp(mass2)
compactness2 = compute_compactness(mass2, radius2)

mej_tot = total_ejecta_mass(mass2, mass1, compactness2, spin1z)
mej_dyn_raw = compute_dynamical_ejecta_mass(mass2, mass1, spin1z, compactness2)
mej_dyn = np.where(mej_dyn_raw > 0, mej_dyn_raw, 0.0)

q = mass2 / mass1
mej_wind = compute_wind_ejecta_mass(mej_tot, mej_dyn, q)

pop = pd.DataFrame(
    {
        'mass1': mass1,
        'mass2': mass2,
        'spin1z': spin1z,
        'spin2z': spin2z,
        'mej_tot': mej_tot,
        'mej_dyn': mej_dyn,
        'mej_wind': mej_wind,
    }
)

pop['kn_positive'] = (
    (pop['mej_dyn'] > MEJ_DYN_THRESHOLD)
    & (pop['mej_wind'] > MEJ_WIND_THRESHOLD)
)

n_pos = int(pop['kn_positive'].sum())
print(f'Candidates: {len(pop):,}')
print(f'mej_tot > 0 count: {(pop["mej_tot"] > 0).sum():,}')
print(f'mej_dyn > 0 count: {(pop["mej_dyn"] > 0).sum():,}')
print(f'mej_wind > 0 count: {(pop["mej_wind"] > 0).sum():,}')
print(f'KN-positive (dyn+wind): {n_pos:,} ({n_pos/len(pop):.2%})')
print(f'mej_tot range: [{pop.mej_tot.min():.3e}, {pop.mej_tot.max():.3e}]')
print(f'mej_dyn range: [{pop.mej_dyn.min():.3e}, {pop.mej_dyn.max():.3e}]')
print(f'mej_wind range: [{pop.mej_wind.min():.3e}, {pop.mej_wind.max():.3e}]')

pop.head()


## Parameter range summary

- `positive_min/max`: support range where KN can appear under `mej_dyn>0 & mej_wind>0`.
- `positive_q05/q95`: robust central range of KN-positive population.


In [ ]:
positive = pop[pop['kn_positive']].copy()

summary = pd.DataFrame(
    {
        'all_min': pop[['mass1', 'mass2', 'spin1z']].min(),
        'all_max': pop[['mass1', 'mass2', 'spin1z']].max(),
        'positive_min': positive[['mass1', 'mass2', 'spin1z']].min(),
        'positive_max': positive[['mass1', 'mass2', 'spin1z']].max(),
        'positive_q05': positive[['mass1', 'mass2', 'spin1z']].quantile(0.05),
        'positive_q95': positive[['mass1', 'mass2', 'spin1z']].quantile(0.95),
    }
)

# Ejecta component summary for KN-positive events
ejecta_summary = positive[['mej_tot', 'mej_dyn', 'mej_wind']].quantile([0.0, 0.05, 0.5, 0.95, 1.0])

summary, ejecta_summary


In [ ]:
# Count KN-positive samples within BULLA-BHNS-M1-2COMP ejecta-mass limits
MEJ_MIN, MEJ_MAX = 0.01, 0.09

in_model_range = (
    positive['mej_dyn'].between(MEJ_MIN, MEJ_MAX, inclusive='both')
    & positive['mej_wind'].between(MEJ_MIN, MEJ_MAX, inclusive='both')
)

n_pos_total = len(positive)
n_pos_in_model = int(in_model_range.sum())
frac_pos_in_model = n_pos_in_model / n_pos_total if n_pos_total > 0 else np.nan

model_limit_stats = pd.DataFrame(
    {
        'value': [
            MEJ_MIN,
            MEJ_MAX,
            n_pos_total,
            n_pos_in_model,
            frac_pos_in_model,
            n_pos_total - n_pos_in_model,
        ]
    },
    index=[
        'mej_lower_bound',
        'mej_upper_bound',
        'n_pos_total',
        'n_pos_in_model_range',
        'frac_pos_in_model_range',
        'n_pos_out_of_model_range',
    ],
)

print(f'BULLA mass limits: mej_dyn/mej_wind in [{MEJ_MIN:.2f}, {MEJ_MAX:.2f}]')
print(f'pos in model range: {n_pos_in_model:,} / {n_pos_total:,} ({frac_pos_in_model:.2%})')
model_limit_stats


## Visualize KN-positive region in parameter planes

Red points are KN-positive (`mej_dyn > 0` and `mej_wind > 0`).


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)

colors = np.where(pop['kn_positive'].values, 'tab:red', 'lightgray')

axes[0].scatter(pop['mass1'], pop['mass2'], c=colors, s=4, alpha=0.5, linewidths=0)
axes[0].set_xlabel('mass1 (BH, Msun)')
axes[0].set_ylabel('mass2 (NS, Msun)')
axes[0].set_title('KN-positive in (mass1, mass2)')
axes[0].grid(alpha=0.25)

axes[1].scatter(pop['mass1'], pop['spin1z'], c=colors, s=4, alpha=0.5, linewidths=0)
axes[1].set_xlabel('mass1 (BH, Msun)')
axes[1].set_ylabel('spin1z (BH aligned spin)')
axes[1].set_title('KN-positive in (mass1, spin1z)')
axes[1].grid(alpha=0.25)

plt.show()


In [ ]:
# Scatter of dynamical vs wind ejecta masses with BULLA allowed ranges
fig, ax = plt.subplots(figsize=(7, 6))

# ax.scatter(pop['mej_dyn'], pop['mej_wind'], c='lightgray', s=4, alpha=0.20, linewidths=0, label='all samples')
ax.scatter(positive['mej_dyn'], positive['mej_wind'], c='tab:red', s=5, alpha=0.75, linewidths=0, label='KN-positive')

dyn_min, dyn_max = 0.01, 0.09
wind_min, wind_max = 0.01, 0.09

ax.axvline(dyn_min, color='tab:blue', linestyle='--', linewidth=1.4)
ax.axvline(dyn_max, color='tab:blue', linestyle='--', linewidth=1.4)
ax.axhline(wind_min, color='tab:blue', linestyle='--', linewidth=1.4)
ax.axhline(wind_max, color='tab:blue', linestyle='--', linewidth=1.4, label='BULLA allowed range')
ax.fill_betweenx([wind_min, wind_max], dyn_min, dyn_max, color='tab:blue', alpha=0.08)

ax.set_xlabel('mej_dyn (dynamical ejecta mass, Msun)')
ax.set_ylabel('mej_wind (wind ejecta mass, Msun)')
ax.set_title('Dynamical vs wind ejecta mass')
ax.set_xlim(left=0.0)
ax.set_ylim(bottom=0.0)
ax.grid(alpha=0.25)
ax.legend(loc='upper right')
plt.show()


In [ ]:
# 2D KN-positive fraction maps
bins_mass1 = np.linspace(BH_MASS_RANGE[0], BH_MASS_RANGE[1], 28)
bins_mass2 = np.linspace(NS_MASS_RANGE[0], NS_MASS_RANGE[1], 28)
bins_spin1 = np.linspace(-BH_SPIN_MAX, BH_SPIN_MAX, 28)

h_pos_m1s1, x_m1s1, y_m1s1 = np.histogram2d(
    pop.loc[pop['kn_positive'], 'mass1'],
    pop.loc[pop['kn_positive'], 'spin1z'],
    bins=[bins_mass1, bins_spin1],
)
h_all_m1s1, _, _ = np.histogram2d(pop['mass1'], pop['spin1z'], bins=[bins_mass1, bins_spin1])
frac_m1s1 = h_pos_m1s1 / np.where(h_all_m1s1 == 0, np.nan, h_all_m1s1)

h_pos_m1m2, x_m1m2, y_m1m2 = np.histogram2d(
    pop.loc[pop['kn_positive'], 'mass1'],
    pop.loc[pop['kn_positive'], 'mass2'],
    bins=[bins_mass1, bins_mass2],
)
h_all_m1m2, _, _ = np.histogram2d(pop['mass1'], pop['mass2'], bins=[bins_mass1, bins_mass2])
frac_m1m2 = h_pos_m1m2 / np.where(h_all_m1m2 == 0, np.nan, h_all_m1m2)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)

im0 = axes[0].pcolormesh(x_m1s1, y_m1s1, frac_m1s1.T, shading='auto', cmap='viridis', vmin=0, vmax=1)
axes[0].set_xlabel('mass1 (BH, Msun)')
axes[0].set_ylabel('spin1z')
axes[0].set_title('KN-positive fraction in (mass1, spin1z)')
cb0 = fig.colorbar(im0, ax=axes[0])
cb0.set_label('positive fraction')

im1 = axes[1].pcolormesh(x_m1m2, y_m1m2, frac_m1m2.T, shading='auto', cmap='viridis', vmin=0, vmax=1)
axes[1].set_xlabel('mass1 (BH, Msun)')
axes[1].set_ylabel('mass2 (NS, Msun)')
axes[1].set_title('KN-positive fraction in (mass1, mass2)')
cb1 = fig.colorbar(im1, ax=axes[1])
cb1.set_label('positive fraction')

plt.show()


## Spin threshold trend vs BH mass

This approximates the minimum aligned spin needed for KN production
under the dual-component criterion (`mej_dyn > 0` and `mej_wind > 0`).


In [ ]:
mass1_bins = np.linspace(BH_MASS_RANGE[0], BH_MASS_RANGE[1], 22)
pop['mass1_bin'] = pd.cut(pop['mass1'], bins=mass1_bins, include_lowest=True)

rows = []
for interval, group in pop.groupby('mass1_bin', observed=True):
    group_pos = group[group['kn_positive']]
    rows.append(
        {
            'mass1_center': 0.5 * (interval.left + interval.right),
            'n_total': len(group),
            'n_positive': len(group_pos),
            'positive_fraction': len(group_pos) / len(group),
            'min_spin_for_kn': group_pos['spin1z'].min() if len(group_pos) else np.nan,
            'median_spin_for_kn': group_pos['spin1z'].median() if len(group_pos) else np.nan,
        }
    )

spin_threshold_by_mass1 = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(
    spin_threshold_by_mass1['mass1_center'],
    spin_threshold_by_mass1['min_spin_for_kn'],
    marker='o',
    linewidth=1.5,
    label='min spin for KN',
)
ax.plot(
    spin_threshold_by_mass1['mass1_center'],
    spin_threshold_by_mass1['median_spin_for_kn'],
    marker='s',
    linewidth=1.5,
    label='median spin for KN',
)
ax.set_xlabel('mass1 (BH, Msun)')
ax.set_ylabel('spin1z')
ax.set_title('Spin requirement trend for KN-positive events')
ax.grid(alpha=0.25)
ax.legend()
plt.show()

spin_threshold_by_mass1


## Optional: export KN-positive population to HDF5

Exported rows follow the dual-component criterion and keep the
`bayestar-inject` compatible columns (`mass1, mass2, spin1z, spin2z`).


In [ ]:
kn_pop = pop.loc[pop['kn_positive'], ['mass1', 'mass2', 'spin1z', 'spin2z']].copy()
print(f'KN-positive rows to export: {len(kn_pop):,}')

if HAS_ASTROPY:
    table = Table.from_pandas(kn_pop.reset_index(drop=True))
    table.write(OUTPUT_H5, overwrite=True)
    print(f'Saved: {OUTPUT_H5.resolve()}')
else:
    print('astropy is not available in this kernel. Skipped writing HDF5.')


## How to use this notebook

1. Adjust `N_CANDIDATES` and ranges in the config cell.
2. If needed, increase `MEJ_DYN_THRESHOLD` / `MEJ_WIND_THRESHOLD` to keep only stronger-ejecta cases.
3. Re-run all cells and inspect `summary`, `ejecta_summary`, heatmaps, and spin-threshold curves.
